In [1]:
# =========================================================
# MUTUAL INFORMATION FEATURE SELECTION — Yelp FM
# Phân tích MI cho 33 features trước khi train
# =========================================================

import json
import numpy as np
import math
from datetime import datetime
from collections import Counter
from sklearn.feature_selection import mutual_info_regression

# =========================================================
# CONFIG
# =========================================================
BASE        = "/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset"
MAX_REVIEWS = None   # dùng 500k để nhanh, tăng lên None nếu muốn full
TOP_CITY    = 50
THRESHOLD   = 0.01      # MI > ngưỡng này thì giữ lại

# =========================================================
# FEATURE NAMES (đồng bộ với fm_pytorch.py / fm_cupy.py)
# =========================================================
FEATURE_NAMES = [
    # USER (10)
    "user_review_count", "user_avg_stars", "user_fans",
    "user_useful",       "user_funny",     "user_cool",
    "user_years_active", "user_elite",     "user_friends", "user_compliments",
    # BIZ (8)
    "biz_stars",       "biz_review_count", "biz_is_open",
    "biz_cat_count",   "biz_open_days",    "biz_price_range",
    "biz_total_hours", "biz_city",
    # TIP USER (3)
    "utip_count", "utip_avg_len", "utip_avg_comp",
    # TIP BIZ (3)
    "btip_count", "btip_avg_len", "btip_avg_comp",
    # CHECKIN (1)
    "checkin_count",
    # REVIEW (4)
    "review_useful", "review_funny", "review_cool", "review_text_len",
    # TEMPORAL (4)
    "year", "month", "weekday", "hour",
]

NUM_DIM = len(FEATURE_NAMES)  # = 33
assert NUM_DIM == 33, f"Expected 33, got {NUM_DIM}"

# =========================================================
# BƯỚC 1 — BUILD CITY MAP
# =========================================================
def build_city_map(business_path, top_n=TOP_CITY):
    counter = Counter()
    with open(business_path, "r", encoding="utf-8") as f:
        for line in f:
            b    = json.loads(line)
            city = (b.get("city") or "Unknown").strip()
            counter[city] += 1
    top_cities = [city for city, _ in counter.most_common(top_n)]
    city_map   = {city: idx + 1 for idx, city in enumerate(top_cities)}
    print(f"  ✓ City map: top {top_n} cities (Other=0)")
    return city_map

# =========================================================
# BƯỚC 2 — LOAD CONTEXTS
# =========================================================
def load_user_context(path):
    ctx = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            u = json.loads(line)
            try:
                dt = datetime.strptime(
                    u.get("yelping_since", "2015-01-01 00:00:00"),
                    "%Y-%m-%d %H:%M:%S")
                years_active = max(0.0, (2020 - dt.year) / 10.0)
            except:
                years_active = 0.5

            elite_str = u.get("elite", "")
            elite_count = (len(elite_str) if isinstance(elite_str, list)
                           else len(elite_str.split(",")) if elite_str else 0)

            friends = u.get("friends", "")
            friend_count = (len([f for f in friends.split(",") if f.strip()])
                            if isinstance(friends, str) and friends and friends != "None"
                            else 0)

            compliment_keys = [
                "compliment_hot","compliment_more","compliment_profile",
                "compliment_cute","compliment_list","compliment_note",
                "compliment_plain","compliment_cool","compliment_funny",
                "compliment_writer","compliment_photos"
            ]
            total_compliments = sum(u.get(k, 0) or 0 for k in compliment_keys)

            ctx[u["user_id"]] = [
                math.log1p(u.get("review_count", 0)) / 10.0,
                u.get("average_stars", 0) / 5.0,
                math.log1p(u.get("fans", 0)) / 10.0,
                math.log1p(max(0, u.get("useful", 0) or 0)) / 10.0,
                math.log1p(max(0, u.get("funny",  0) or 0)) / 10.0,
                math.log1p(max(0, u.get("cool",   0) or 0)) / 10.0,
                years_active,
                math.log1p(elite_count) / 10.0,
                math.log1p(friend_count) / 10.0,
                math.log1p(total_compliments) / 10.0,
            ]
    print(f"  ✓ Users: {len(ctx):,}")
    return ctx


def load_business_context(path, city_map):
    ctx = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            b          = json.loads(line)
            categories = b.get("categories", "") or ""
            cat_count  = len(categories.split(",")) if categories else 0

            hours       = b.get("hours", {}) or {}
            open_days   = len(hours)
            total_hours = 0.0
            for day, hrs in hours.items():
                if not hrs or hrs == "0:0-0:0":
                    continue
                try:
                    open_t, close_t = hrs.split("-")
                    oh, om = map(int, open_t.split(":"))
                    ch, cm = map(int, close_t.split(":"))
                    total_hours += max(0, (ch*60+cm) - (oh*60+om)) / 60.0
                except:
                    total_hours += 8.0

            attrs     = b.get("attributes", {}) or {}
            price_raw = attrs.get("RestaurantsPriceRange2", 0) or 0
            try:
                price_norm = float(price_raw) / 4.0
            except:
                price_norm = 0.0

            city      = (b.get("city") or "Unknown").strip()
            city_norm = city_map.get(city, 0) / TOP_CITY

            ctx[b["business_id"]] = [
                b.get("stars", 0) / 5.0,
                math.log1p(b.get("review_count", 0)) / 10.0,
                float(b.get("is_open", 0)),
                math.log1p(cat_count) / 10.0,
                open_days / 7.0,
                price_norm,
                min(total_hours, 168.0) / 168.0,
                city_norm,
            ]
    print(f"  ✓ Businesses: {len(ctx):,}")
    return ctx


def load_checkin_context(path):
    ctx = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            c   = json.loads(line)
            bid = c["business_id"]
            dates = c.get("date", "")
            ctx[bid] = math.log1p(len(dates.split(", ")) if dates else 0) / 10.0
    print(f"  ✓ Checkin: {len(ctx):,} businesses")
    return ctx


def load_tip_context(path):
    user_tip, biz_tip = {}, {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            t    = json.loads(line)
            uid  = t["user_id"]
            bid  = t["business_id"]
            length = len(t.get("text", ""))
            comp   = max(0, t.get("compliment_count", 0) or 0)
            for d, key in [(user_tip, uid), (biz_tip, bid)]:
                if key not in d:
                    d[key] = [0, 0, 0]
                d[key][0] += 1
                d[key][1] += length
                d[key][2] += comp
    print(f"  ✓ Tips: {len(user_tip):,} users, {len(biz_tip):,} businesses")
    return user_tip, biz_tip

# =========================================================
# BƯỚC 3 — EXTRACT FEATURES TỪ REVIEWS
# =========================================================
def extract_features(review_path, user_ctx, biz_ctx,
                     checkin_ctx, user_tip, biz_tip,
                     max_reviews=None):
    print(f"\nExtracting features from reviews...")
    num_features = []
    ratings      = []
    count        = 0

    with open(review_path, "r", encoding="utf-8") as f:
        for line in f:
            if max_reviews and count >= max_reviews:
                break

            r   = json.loads(line)
            uid = r["user_id"]
            bid = r["business_id"]

            uctx = user_ctx.get(uid, [0.0] * 10)
            bctx = biz_ctx.get(bid, [0.0] * 8)

            utip = user_tip.get(uid, [0, 0, 0])
            utip_feat = [
                math.log1p(utip[0]) / 10.0,
                math.log1p(utip[1] / max(utip[0], 1)) / 10.0,
                math.log1p(utip[2] / max(utip[0], 1)) / 10.0,
            ]

            btip = biz_tip.get(bid, [0, 0, 0])
            btip_feat = [
                math.log1p(btip[0]) / 10.0,
                math.log1p(btip[1] / max(btip[0], 1)) / 10.0,
                math.log1p(btip[2] / max(btip[0], 1)) / 10.0,
            ]

            checkin_feat = [checkin_ctx.get(bid, 0.0)]

            useful   = max(0, int(r.get("useful", 0) or 0))
            funny    = max(0, int(r.get("funny",  0) or 0))
            cool     = max(0, int(r.get("cool",   0) or 0))
            text_len = len(r.get("text", ""))
            review_feat = [
                math.log1p(useful)   / 10.0,
                math.log1p(funny)    / 10.0,
                math.log1p(cool)     / 10.0,
                math.log1p(text_len) / 10.0,
            ]

            dt = datetime.strptime(r["date"], "%Y-%m-%d %H:%M:%S")
            temporal_feat = [
                (dt.year - 2015) / 10.0,
                dt.month    / 12.0,
                dt.weekday() / 6.0,
                dt.hour     / 23.0,
            ]

            row = np.array(
                uctx + bctx + utip_feat + btip_feat +
                checkin_feat + review_feat + temporal_feat,
                dtype=np.float32
            )

            assert len(row) == NUM_DIM, f"Dim mismatch: {len(row)} != {NUM_DIM}"

            num_features.append(row)
            ratings.append(float(r["stars"]))
            count += 1

            if count % 100_000 == 0:
                print(f"  Loaded {count:,} reviews...")

    X = np.array(num_features, dtype=np.float32)
    y = np.array(ratings,      dtype=np.float32)
    print(f"  ✓ Total: {count:,} reviews | Shape: {X.shape}")
    return X, y

# =========================================================
# BƯỚC 4 — MUTUAL INFORMATION
# =========================================================
def run_mutual_info(X, y, feature_names, threshold=THRESHOLD):
    print(f"\nRunning Mutual Information (n_samples={len(X):,})...")
    print("  (Có thể mất 2-5 phút tùy số lượng reviews)")

    # mutual_info_regression: phù hợp vì rating là liên tục (1.0-5.0)
    mi_scores = mutual_info_regression(
        X, y,
        random_state=42,
        n_neighbors=5,
    )

    # Sắp xếp kết quả
    results = sorted(
        zip(feature_names, mi_scores),
        key=lambda x: x[1],
        reverse=True
    )

    # In kết quả
    print(f"\n{'='*70}")
    print(f"MUTUAL INFORMATION — {len(feature_names)} Features")
    print(f"{'='*70}")
    print(f"\n{'Rank':<5} {'Feature':<25} {'MI Score':>10}  {'Bar':<30} Đánh giá")
    print("-" * 80)

    for i, (name, score) in enumerate(results, 1):
        bar  = "█" * min(int(score * 300), 30)
        tag  = ("🔥 Cao"  if score > 0.05
                else "⚠  TB"   if score > 0.01
                else "❌ Thấp")
        print(f"{i:<5} {name:<25} {score:>10.4f}  {bar:<30} {tag}")

    # Phân nhóm
    high   = [(n, s) for n, s in results if s > 0.05]
    medium = [(n, s) for n, s in results if 0.01 < s <= 0.05]
    low    = [(n, s) for n, s in results if s <= 0.01]

    print(f"\n{'='*70}")
    print(f"PHÂN NHÓM THEO MI SCORE")
    print(f"{'='*70}")
    print(f"\n🔥 Cao (MI > 0.05) — {len(high)} features:")
    for n, s in high:
        print(f"   + {n:<25} {s:.4f}")

    print(f"\n⚠  Trung bình (0.01 < MI ≤ 0.05) — {len(medium)} features:")
    for n, s in medium:
        print(f"   ~ {n:<25} {s:.4f}")

    print(f"\n❌ Thấp (MI ≤ 0.01) — {len(low)} features:")
    for n, s in low:
        print(f"   - {n:<25} {s:.4f}")

    # Gợi ý
    kept    = [n for n, s in results if s > threshold]
    dropped = [n for n, s in results if s <= threshold]

    print(f"\n{'='*70}")
    print(f"GỢI Ý (threshold = {threshold})")
    print(f"{'='*70}")
    print(f"\n✅ Giữ lại ({len(kept)} features):")
    for n in kept:
        s = dict(results)[n]
        print(f"   + {n:<25} MI={s:.4f}")

    if dropped:
        print(f"\n❌ Xem xét bỏ ({len(dropped)} features):")
        for n in dropped:
            s = dict(results)[n]
            print(f"   - {n:<25} MI={s:.4f}")
    else:
        print(f"\n✓ Không có feature nào dưới ngưỡng {threshold}")

    return results, kept, dropped

# =========================================================
# BƯỚC 5 — SO SÁNH VỚI PERMUTATION IMPORTANCE
# =========================================================
def compare_with_permutation(mi_results, perm_results=None):
    """
    perm_results: list of (name, rmse_change) từ permutation_importance()
    Nếu chưa có thì bỏ qua
    """
    if perm_results is None:
        print("\n(Chưa có Permutation Importance — chạy sau khi train)")
        return

    mi_dict   = {n: s for n, s in mi_results}
    perm_dict = {n: s for n, s in perm_results}

    print(f"\n{'='*70}")
    print("SO SÁNH MI vs PERMUTATION IMPORTANCE")
    print(f"{'='*70}")
    print(f"\n{'Feature':<25} {'MI Score':>10} {'Perm RMSE+':>12}  Nhận xét")
    print("-" * 60)

    all_names = list(mi_dict.keys())
    for name in sorted(all_names, key=lambda n: mi_dict[n], reverse=True):
        mi   = mi_dict.get(name, 0)
        perm = perm_dict.get(name, 0)

        if mi > 0.01 and perm > 0.005:
            note = "✓ Cả 2 đều đồng ý quan trọng"
        elif mi <= 0.01 and perm <= 0.005:
            note = "✗ Cả 2 đều đồng ý yếu"
        elif mi > 0.01 and perm <= 0.005:
            note = "⚠ MI cao nhưng Perm thấp — ít ảnh hưởng model"
        else:
            note = "⚠ MI thấp nhưng Perm cao — tương tác quan trọng"

        print(f"{name:<25} {mi:>10.4f} {perm:>+12.4f}  {note}")

# =========================================================
# SAVE KẾT QUẢ
# =========================================================
def save_results(mi_results, kept, dropped, output_path):
    import json as _json
    output = {
        "config": {
            "max_reviews": MAX_REVIEWS,
            "threshold":   THRESHOLD,
            "top_city":    TOP_CITY,
            "num_features": NUM_DIM,
        },
        "mi_scores": [
            {"feature": n, "mi_score": float(s)}
            for n, s in mi_results
        ],
        "kept_features":    kept,
        "dropped_features": dropped,
    }
    with open(output_path, "w") as f:
        _json.dump(output, f, indent=2)
    print(f"\n✓ Saved to {output_path}")

# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    print("=" * 70)
    print("MUTUAL INFORMATION FEATURE ANALYSIS — Yelp Dataset")
    print("=" * 70)

    # Bước 1: Build city map
    print("\nBước 1: Build city map")
    city_map = build_city_map(f"{BASE}/yelp_academic_dataset_business.json")

    # Bước 2: Load contexts
    print("\nBước 2: Load contexts")
    user_ctx          = load_user_context(f"{BASE}/yelp_academic_dataset_user.json")
    biz_ctx           = load_business_context(
                            f"{BASE}/yelp_academic_dataset_business.json", city_map)
    checkin_ctx       = load_checkin_context(f"{BASE}/yelp_academic_dataset_checkin.json")
    user_tip, biz_tip = load_tip_context(f"{BASE}/yelp_academic_dataset_tip.json")

    # Bước 3: Extract features
    print("\nBước 3: Extract features")
    X, y = extract_features(
        review_path  = f"{BASE}/yelp_academic_dataset_review.json",
        user_ctx     = user_ctx,
        biz_ctx      = biz_ctx,
        checkin_ctx  = checkin_ctx,
        user_tip     = user_tip,
        biz_tip      = biz_tip,
        max_reviews  = MAX_REVIEWS,
    )

    # Bước 4: Mutual Information
    print("\nBước 4: Mutual Information")
    mi_results, kept, dropped = run_mutual_info(X, y, FEATURE_NAMES, THRESHOLD)

    # Bước 5: So sánh (điền perm_results sau khi train)
    # Ví dụ nếu đã có permutation results:
    # compare_with_permutation(mi_results, perm_results)
    compare_with_permutation(mi_results, perm_results=None)

    # Save
    import os
    os.makedirs("/kaggle/working", exist_ok=True)
    save_results(mi_results, kept, dropped,
                 "/kaggle/working/mi_feature_selection.json")

    print("\n" + "="*70)
    print("XONG! Các bước tiếp theo:")
    print("  1. Xem file /kaggle/working/mi_feature_selection.json")
    print("  2. Train FM với fm_pytorch.py hoặc fm_cupy.py")
    print("  3. Copy perm_results từ output train vào compare_with_permutation()")
    print("  4. So sánh MI vs Permutation để kết luận cho luận văn")
    print("="*70)

MUTUAL INFORMATION FEATURE ANALYSIS — Yelp Dataset

Bước 1: Build city map
  ✓ City map: top 50 cities (Other=0)

Bước 2: Load contexts
  ✓ Users: 1,987,897
  ✓ Businesses: 150,346
  ✓ Checkin: 131,930 businesses
  ✓ Tips: 301,758 users, 106,193 businesses

Bước 3: Extract features

Extracting features from reviews...
  Loaded 100,000 reviews...
  Loaded 200,000 reviews...
  Loaded 300,000 reviews...
  Loaded 400,000 reviews...
  Loaded 500,000 reviews...
  Loaded 600,000 reviews...
  Loaded 700,000 reviews...
  Loaded 800,000 reviews...
  Loaded 900,000 reviews...
  Loaded 1,000,000 reviews...
  Loaded 1,100,000 reviews...
  Loaded 1,200,000 reviews...
  Loaded 1,300,000 reviews...
  Loaded 1,400,000 reviews...
  Loaded 1,500,000 reviews...
  Loaded 1,600,000 reviews...
  Loaded 1,700,000 reviews...
  Loaded 1,800,000 reviews...
  Loaded 1,900,000 reviews...
  Loaded 2,000,000 reviews...
  Loaded 2,100,000 reviews...
  Loaded 2,200,000 reviews...
  Loaded 2,300,000 reviews...
  Loaded